# Q-Mesh: Physical Cryptography — Zero-Interaction Authentication

**Zipminator PQC Platform** | Notebook 08

---

This notebook demonstrates how WiFi CSI sensing from [RuView](https://github.com/MoHoushmand/RuView) creates a new category of security where the physical world IS the authentication system. We visualize:

- 3D Gaussian splatting of WiFi electromagnetic fields
- CSI subcarrier analysis and entropy extraction
- Biometric detection (breathing, heart rate) from WiFi signals
- PUEK (Physical Unclonable Environment Key) enrollment and verification
- Clearance level enforcement (L1-L4)
- EM Canary threat detection
- Full zero-interaction authentication pipeline

> **Disclaimer:** All data in this notebook is synthetically generated to demonstrate the Physical Cryptography concepts. Real CSI data requires RuView ESP32-S3 hardware.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# ── Zipminator Quantum Dark Theme ──
ZM_STYLE = {
    'figure.facecolor': '#0a0f1e',
    'axes.facecolor': '#111827',
    'axes.edgecolor': '#334155',
    'axes.labelcolor': '#e2e8f0',
    'axes.titleweight': 'bold',
    'axes.grid': True,
    'grid.color': '#1e293b',
    'grid.alpha': 0.5,
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'legend.facecolor': '#1e293b',
    'legend.edgecolor': '#334155',
    'font.family': 'sans-serif',
    'font.size': 11,
    'savefig.facecolor': '#0a0f1e',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
}
plt.rcParams.update(ZM_STYLE)

# ── Quantum Color Palette ──
ZM_CYAN    = '#22d3ee'
ZM_VIOLET  = '#a78bfa'
ZM_EMERALD = '#34d399'
ZM_AMBER   = '#f59e0b'
ZM_ROSE    = '#fb7185'
ZM_BLUE    = '#3b82f6'

# Reproducible synthetic data
np.random.seed(42)

print("Quantum dark theme loaded.")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"\nNotebook 08: Q-Mesh Physical Cryptography")
print("All data is synthetically generated for demonstration purposes.")

## 1. What is Gaussian Splatting for WiFi CSI?

3D Gaussian splatting (originally from NeRF research) represents a scene as a
collection of 3D Gaussian functions. Each Gaussian has a position, covariance
(shape/orientation), opacity, and color. When rendered, they "splat" onto the
viewing plane to reconstruct a continuous field.

When applied to WiFi Channel State Information (CSI), each Gaussian represents
a region of electromagnetic interaction:

- **Wall reflections**: elongated Gaussians aligned along surfaces where WiFi bounces
- **Furniture scattering**: compact Gaussians at positions where objects diffuse the signal
- **Human body absorption**: large, soft Gaussians where the human body absorbs and re-radiates RF energy

The key insight: the Gaussian splat pattern changes dramatically when people
enter, move, or leave a room. The "shape" of the electromagnetic field is a
fingerprint of the room's physical contents. Since a human body is 60% water
(a strong RF absorber at 2.4/5 GHz), a person's presence creates unmistakable
perturbations in the CSI eigenstructure.

This makes the invisible (WiFi signals bouncing through space) visible and,
more importantly, cryptographically useful.

In [ ]:
def generate_room_gaussians(include_people=False, seed=42):
    """Generate 3D Gaussian splat positions for a room's EM field."""
    rng = np.random.RandomState(seed)
    gaussians = {'x': [], 'y': [], 'z': [], 's': [], 'c': [], 'a': []}

    # Wall reflections: elongated along walls (6m x 4m x 3m room)
    for wall_y in [0.0, 4.0]:
        n = 40
        gaussians['x'].extend(np.linspace(0.3, 5.7, n))
        gaussians['y'].extend(np.full(n, wall_y) + rng.normal(0, 0.08, n))
        gaussians['z'].extend(rng.uniform(0.5, 2.5, n))
        gaussians['s'].extend(rng.uniform(15, 40, n))
        gaussians['c'].extend([ZM_VIOLET] * n)
        gaussians['a'].extend(rng.uniform(0.15, 0.35, n))

    for wall_x in [0.0, 6.0]:
        n = 30
        gaussians['x'].extend(np.full(n, wall_x) + rng.normal(0, 0.08, n))
        gaussians['y'].extend(np.linspace(0.3, 3.7, n))
        gaussians['z'].extend(rng.uniform(0.5, 2.5, n))
        gaussians['s'].extend(rng.uniform(15, 40, n))
        gaussians['c'].extend([ZM_VIOLET] * n)
        gaussians['a'].extend(rng.uniform(0.15, 0.35, n))

    # Floor and ceiling reflections
    for z_plane in [0.0, 3.0]:
        n = 25
        gaussians['x'].extend(rng.uniform(0.5, 5.5, n))
        gaussians['y'].extend(rng.uniform(0.5, 3.5, n))
        gaussians['z'].extend(np.full(n, z_plane) + rng.normal(0, 0.05, n))
        gaussians['s'].extend(rng.uniform(10, 25, n))
        gaussians['c'].extend([ZM_BLUE] * n)
        gaussians['a'].extend(rng.uniform(0.08, 0.2, n))

    # Furniture scattering (desk, bookshelf, chair)
    furniture_positions = [(1.5, 1.0, 0.8), (4.0, 3.0, 0.5), (2.5, 2.5, 0.4),
                          (4.5, 1.5, 1.2), (1.0, 3.2, 0.6)]
    for fx, fy, fz in furniture_positions:
        n = 8
        gaussians['x'].extend(fx + rng.normal(0, 0.2, n))
        gaussians['y'].extend(fy + rng.normal(0, 0.2, n))
        gaussians['z'].extend(fz + rng.normal(0, 0.15, n))
        gaussians['s'].extend(rng.uniform(20, 50, n))
        gaussians['c'].extend(['#1e40af'] * n)
        gaussians['a'].extend(rng.uniform(0.1, 0.3, n))

    # Ambient multipath scatter (diffuse field)
    n = 50
    gaussians['x'].extend(rng.uniform(0.5, 5.5, n))
    gaussians['y'].extend(rng.uniform(0.5, 3.5, n))
    gaussians['z'].extend(rng.uniform(0.3, 2.7, n))
    gaussians['s'].extend(rng.uniform(5, 15, n))
    gaussians['c'].extend(['#334155'] * n)
    gaussians['a'].extend(rng.uniform(0.03, 0.1, n))

    if include_people:
        # Person A at (2.0, 2.0) - standing
        n = 20
        gaussians['x'].extend(2.0 + rng.normal(0, 0.15, n))
        gaussians['y'].extend(2.0 + rng.normal(0, 0.12, n))
        gaussians['z'].extend(rng.uniform(0.3, 1.8, n))
        gaussians['s'].extend(rng.uniform(40, 100, n))
        gaussians['c'].extend([ZM_AMBER] * n)
        gaussians['a'].extend(rng.uniform(0.25, 0.55, n))

        # Person B at (4.0, 1.5) - seated
        n = 15
        gaussians['x'].extend(4.0 + rng.normal(0, 0.15, n))
        gaussians['y'].extend(1.5 + rng.normal(0, 0.12, n))
        gaussians['z'].extend(rng.uniform(0.3, 1.2, n))
        gaussians['s'].extend(rng.uniform(35, 85, n))
        gaussians['c'].extend([ZM_ROSE] * n)
        gaussians['a'].extend(rng.uniform(0.25, 0.55, n))

        # Additional multipath perturbations caused by bodies
        n = 30
        gaussians['x'].extend(rng.uniform(1.0, 5.0, n))
        gaussians['y'].extend(rng.uniform(0.5, 3.5, n))
        gaussians['z'].extend(rng.uniform(0.5, 2.0, n))
        gaussians['s'].extend(rng.uniform(8, 25, n))
        gaussians['c'].extend([ZM_AMBER if i % 2 == 0 else ZM_ROSE for i in range(n)])
        gaussians['a'].extend(rng.uniform(0.05, 0.2, n))

    return {k: np.array(v) if k != 'c' else v for k, v in gaussians.items()}


fig = plt.figure(figsize=(18, 8))
fig.suptitle("WiFi CSI Gaussian Splatting: The Room's Electromagnetic Fingerprint",
             fontsize=16, fontweight='bold', color=ZM_CYAN, y=0.98)

for idx, (title, people) in enumerate([("Empty Room", False),
                                        ("Room with 2 People", True)]):
    ax = fig.add_subplot(1, 2, idx + 1, projection='3d')
    ax.set_facecolor('#111827')
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('#1e293b')
    ax.yaxis.pane.set_edgecolor('#1e293b')
    ax.zaxis.pane.set_edgecolor('#1e293b')
    ax.grid(True, alpha=0.2, color='#334155')

    g = generate_room_gaussians(include_people=people, seed=42 + idx)

    ax.scatter(g['x'], g['y'], g['z'], s=g['s'], c=g['c'],
               alpha=np.clip(g['a'], 0.05, 0.6), edgecolors='none')

    # ESP32 nodes at corners (elevated)
    nodes_x = [0.2, 5.8, 0.2, 5.8]
    nodes_y = [0.2, 0.2, 3.8, 3.8]
    nodes_z = [2.8, 2.8, 2.8, 2.8]
    ax.scatter(nodes_x, nodes_y, nodes_z, s=120, c=ZM_CYAN,
               marker='D', edgecolors='white', linewidths=0.8, zorder=10,
               label='RuView ESP32 Nodes')

    if people:
        ax.scatter([2.0], [2.0], [0.9], s=200, c=ZM_AMBER, marker='o',
                   edgecolors='white', linewidths=1, zorder=10, label='Person A (standing)')
        ax.scatter([4.0], [1.5], [0.6], s=180, c=ZM_ROSE, marker='o',
                   edgecolors='white', linewidths=1, zorder=10, label='Person B (seated)')

    ax.set_xlim(0, 6)
    ax.set_ylim(0, 4)
    ax.set_zlim(0, 3)
    ax.set_xlabel('X (m)', fontsize=9, labelpad=5)
    ax.set_ylabel('Y (m)', fontsize=9, labelpad=5)
    ax.set_zlabel('Z (m)', fontsize=9, labelpad=5)
    ax.set_title(title, fontsize=13, pad=10)
    ax.view_init(elev=25, azim=-45)
    ax.legend(fontsize=8, loc='upper left', markerscale=0.7)
    ax.tick_params(labelsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

g_empty = generate_room_gaussians(False)
g_people = generate_room_gaussians(True)
print(f"\nGaussian splat counts:")
print(f"  Empty room:      {len(g_empty['x']):>4} Gaussians")
print(f"  With 2 people:   {len(g_people['x']):>4} Gaussians")
print(f"  Delta:           {len(g_people['x']) - len(g_empty['x']):>4} (body absorption + multipath perturbation)")

## 2. CSI Subcarrier Analysis

Each WiFi frame (802.11n/ac/ax) contains 56 complex subcarrier values in the
Channel State Information. The amplitude and phase of each subcarrier encode
the multipath environment between transmitter and receiver:

$$H(f_k, t) = |H(f_k, t)| \cdot e^{j\phi(f_k, t)}$$

where $H(f_k, t)$ is the channel response at subcarrier frequency $f_k$ and
time $t$.

When a person breathes, their chest wall moves by approximately 1 cm, creating
periodic phase shifts at the breathing frequency (0.2-0.33 Hz, or 12-20
breaths/min). The phase shift on subcarrier $k$ is:

$$\Delta\phi_k(t) = \frac{4\pi \Delta d(t)}{\lambda_k}$$

where $\Delta d(t)$ is the path length change and $\lambda_k$ is the wavelength
at subcarrier $k$. At 5 GHz ($\lambda \approx 6$ cm), a 1 cm chest
displacement produces a phase shift of approximately
$\frac{4\pi \times 0.01}{0.06} \approx 2.1$ radians, which is easily
measurable.

Heart rate creates even smaller motions (~0.1 mm) visible in the CSI
micro-Doppler at 1-2 Hz. These signals require bandpass filtering and
higher-order statistics to extract reliably, but the ESP32-S3's 20 Hz CSI
sampling rate provides sufficient Nyquist margin.

In [ ]:
# ── Generate synthetic CSI data ──
N_SUBCARRIERS = 56
N_FRAMES = 200
FS = 20  # Hz sampling rate
T_TOTAL = N_FRAMES / FS  # 10 seconds

t = np.arange(N_FRAMES) / FS
subcarrier_idx = np.arange(N_SUBCARRIERS)

# Base phase: stable room structure (random but constant per subcarrier)
base_phase = np.random.uniform(-np.pi, np.pi, (N_SUBCARRIERS, 1))
base_phase = np.tile(base_phase, (1, N_FRAMES))

# Breathing modulation: 0.3 Hz (18 breaths/min), affects subcarriers 10-30
breathing_freq = 0.3  # Hz
breathing_amp = 0.8   # radians
breathing_mask = np.zeros(N_SUBCARRIERS)
breathing_mask[10:31] = np.hanning(21)  # Windowed to body-facing subcarriers
breathing_signal = breathing_amp * np.sin(2 * np.pi * breathing_freq * t)
breathing_component = np.outer(breathing_mask, breathing_signal)

# Heartbeat modulation: 1.2 Hz (72 BPM), smaller amplitude, subcarriers 15-25
heartbeat_freq = 1.2  # Hz
heartbeat_amp = 0.12  # radians (much smaller than breathing)
heartbeat_mask = np.zeros(N_SUBCARRIERS)
heartbeat_mask[15:26] = np.hanning(11)
heartbeat_signal = heartbeat_amp * np.sin(2 * np.pi * heartbeat_freq * t)
heartbeat_component = np.outer(heartbeat_mask, heartbeat_signal)

# Measurement noise
noise = np.random.normal(0, 0.05, (N_SUBCARRIERS, N_FRAMES))

# Combined CSI phase matrix
csi_phase = base_phase + breathing_component + heartbeat_component + noise
csi_phase = np.angle(np.exp(1j * csi_phase))  # Wrap to [-pi, pi]

# ── Von Neumann entropy extraction per frame ──
def von_neumann_entropy(phase_vector):
    """Estimate entropy bits extractable from a CSI phase vector."""
    bins = np.digitize(phase_vector, np.linspace(-np.pi, np.pi, 9)) - 1
    bins = np.clip(bins, 0, 7)
    counts = np.bincount(bins, minlength=8)
    probs = counts / len(phase_vector)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

entropy_per_frame = [von_neumann_entropy(csi_phase[:, i]) for i in range(N_FRAMES)]

# ── Visualization: 2x2 layout ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("CSI Subcarrier Phase Analysis: Extracting Biometrics from WiFi",
             fontsize=16, fontweight='bold', color=ZM_CYAN, y=0.98)

# Panel 1: CSI phase heatmap
ax = axes[0, 0]
im = ax.pcolormesh(t, subcarrier_idx, csi_phase, cmap='twilight_shifted',
                   shading='auto', vmin=-np.pi, vmax=np.pi)
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Subcarrier Index', fontsize=11)
ax.set_title('CSI Phase Across All Subcarriers', fontsize=13)
cb = plt.colorbar(im, ax=ax, label='Phase (rad)', pad=0.02)
cb.ax.yaxis.label.set_color('#e2e8f0')
cb.ax.tick_params(colors='#94a3b8')
ax.axhline(y=10, color=ZM_AMBER, linestyle='--', alpha=0.5, linewidth=0.8)
ax.axhline(y=30, color=ZM_AMBER, linestyle='--', alpha=0.5, linewidth=0.8)
ax.text(0.3, 32, 'Body-facing band', fontsize=8, color=ZM_AMBER)

# Panel 2: Single subcarrier phase over time
ax = axes[0, 1]
sc_idx = 20
phase_sc20 = csi_phase[sc_idx, :] - csi_phase[sc_idx, 0]
ax.plot(t, phase_sc20, color=ZM_CYAN, linewidth=1.2, alpha=0.9,
        label=f'Subcarrier #{sc_idx}')
ax.plot(t, breathing_amp * np.sin(2 * np.pi * breathing_freq * t) * breathing_mask[sc_idx],
        color=ZM_AMBER, linewidth=1.5, linestyle='--', alpha=0.7,
        label='Breathing component')
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Phase Change (rad)', fontsize=11)
ax.set_title(f'Subcarrier #{sc_idx}: Breathing + Heartbeat', fontsize=13)
ax.legend(fontsize=9)

# Panel 3: FFT of subcarrier #20
ax = axes[1, 0]
phase_detrended = phase_sc20 - np.mean(phase_sc20)
freqs = np.fft.rfftfreq(N_FRAMES, d=1/FS)
fft_mag = np.abs(np.fft.rfft(phase_detrended * np.hanning(N_FRAMES)))
fft_mag_db = 20 * np.log10(fft_mag / np.max(fft_mag) + 1e-10)

ax.plot(freqs, fft_mag_db, color=ZM_VIOLET, linewidth=1.5)
ax.axvline(x=breathing_freq, color=ZM_AMBER, linestyle='--', alpha=0.8,
           label=f'Breathing: {breathing_freq} Hz ({breathing_freq*60:.0f} BPM)')
ax.axvline(x=heartbeat_freq, color=ZM_ROSE, linestyle='--', alpha=0.8,
           label=f'Heartbeat: {heartbeat_freq} Hz ({heartbeat_freq*60:.0f} BPM)')
ax.set_xlabel('Frequency (Hz)', fontsize=11)
ax.set_ylabel('Magnitude (dB)', fontsize=11)
ax.set_title(f'FFT of Subcarrier #{sc_idx}', fontsize=13)
ax.set_xlim(0, FS/2)
ax.set_ylim(-60, 5)
ax.legend(fontsize=9)

# Panel 4: Von Neumann entropy per frame
ax = axes[1, 1]
ax.plot(t, entropy_per_frame, color=ZM_EMERALD, linewidth=1.5, alpha=0.9)
ax.axhline(y=3.0, color=ZM_AMBER, linestyle='--', alpha=0.6,
           label='Max entropy (3 bits for 8 bins)')
ax.fill_between(t, entropy_per_frame, alpha=0.15, color=ZM_EMERALD)
mean_ent = np.mean(entropy_per_frame)
ax.axhline(y=mean_ent, color=ZM_CYAN, linestyle=':', alpha=0.6,
           label=f'Mean: {mean_ent:.2f} bits/frame')
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Entropy (bits)', fontsize=11)
ax.set_title('Von Neumann Entropy Extraction', fontsize=13)
ax.set_ylim(1.5, 3.2)
ax.legend(fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print(f"\nCSI Analysis Summary:")
print(f"  Subcarriers:       {N_SUBCARRIERS}")
print(f"  Frames captured:   {N_FRAMES} ({T_TOTAL:.0f}s at {FS} Hz)")
print(f"  Breathing rate:    {breathing_freq * 60:.0f} breaths/min ({breathing_freq} Hz)")
print(f"  Heart rate:        {heartbeat_freq * 60:.0f} BPM ({heartbeat_freq} Hz)")
print(f"  Mean entropy:      {mean_ent:.3f} bits/frame")
print(f"  Total extractable: {mean_ent * N_FRAMES:.0f} bits over {T_TOTAL:.0f}s")

## 3. Biometric Profile Detection

Each person has a unique "biometric shape" derived from their interaction with
WiFi signals. The combination of:

- **Respiratory pattern**: rate, depth, regularity (chest wall displacement)
- **Cardiac rhythm**: resting heart rate, HRV (inter-beat interval variation)
- **Postural micro-movements**: sway, fidgeting, tremor

creates a fingerprint that cannot be forged because it comes from the physics
of a specific human body interacting with electromagnetic waves. Unlike a
password (knowledge-based) or a fingerprint scan (possession-based), this
biometric is:

1. **Continuous**: verified every frame, not just at login
2. **Contactless**: no reader, no sensor on the body
3. **Coercion-aware**: stress changes breathing and heart rate, enabling duress detection
4. **Non-replayable**: the biometric includes the room's EM eigenstructure, so a recording from another location fails verification

This is what separates Q-Mesh from traditional biometrics. A fingerprint can
be lifted from a glass. A face can be deepfaked. But you cannot fake the
electromagnetic interaction between a specific human body and a specific room
at a specific moment in time.

In [ ]:
# ── Biometric profiles for 3 subjects ──
profiles = {
    'Person A (calm)': {
        'breathing_rate': 14,
        'heart_rate': 68,
        'breathing_regularity': 0.92,
        'hrv': 0.85,
        'movement_amplitude': 0.15,
        'movement_regularity': 0.88,
        'color': ZM_CYAN,
    },
    'Person B (active)': {
        'breathing_rate': 18,
        'heart_rate': 82,
        'breathing_regularity': 0.78,
        'hrv': 0.72,
        'movement_amplitude': 0.45,
        'movement_regularity': 0.65,
        'color': ZM_VIOLET,
    },
    'Person C (stress)': {
        'breathing_rate': 22,
        'heart_rate': 105,
        'breathing_regularity': 0.45,
        'hrv': 0.38,
        'movement_amplitude': 0.72,
        'movement_regularity': 0.35,
        'color': ZM_ROSE,
    },
}

# Normalize to [0, 1] for radar chart
axes_labels = ['Breathing\nRate', 'Heart\nRate', 'Breathing\nRegularity',
               'HRV', 'Movement\nAmplitude', 'Movement\nRegularity']
axes_keys = ['breathing_rate', 'heart_rate', 'breathing_regularity',
             'hrv', 'movement_amplitude', 'movement_regularity']

norm_ranges = {
    'breathing_rate': (10, 30),
    'heart_rate': (50, 120),
    'breathing_regularity': (0, 1),
    'hrv': (0, 1),
    'movement_amplitude': (0, 1),
    'movement_regularity': (0, 1),
}

def normalize(value, key):
    lo, hi = norm_ranges[key]
    return (value - lo) / (hi - lo)

# ── Radar chart ──
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig.suptitle("WiFi CSI Biometric Profiles: Three Subjects",
             fontsize=16, fontweight='bold', color=ZM_CYAN, y=0.98)

N_AXES = len(axes_labels)
angles = np.linspace(0, 2 * np.pi, N_AXES, endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

ax.set_facecolor('#111827')

for name, profile in profiles.items():
    values = [normalize(profile[k], k) for k in axes_keys]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, markersize=6,
            color=profile['color'], label=name, alpha=0.9)
    ax.fill(angles, values, alpha=0.12, color=profile['color'])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(axes_labels, fontsize=10, color='#e2e8f0')
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8, color='#94a3b8')
ax.spines['polar'].set_color('#334155')
ax.grid(color='#1e293b', alpha=0.5)

# Duress detection annotation
ax.annotate('DURESS DETECTION ZONE\nL3+ triggers alert',
            xy=(angles[0], 0.85), fontsize=9, fontweight='bold',
            color=ZM_ROSE, ha='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#1e293b',
                      edgecolor=ZM_ROSE, alpha=0.8))

ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print("\nBiometric Profile Summary:")
print(f"{'Subject':<22} {'BR':>5} {'HR':>5} {'BR-Reg':>7} {'HRV':>5} "
      f"{'Mov-A':>6} {'Mov-R':>6} {'Status':>10}")
print("-" * 72)
for name, p in profiles.items():
    status = 'DURESS' if p['heart_rate'] > 100 and p['breathing_regularity'] < 0.5 else 'NORMAL'
    print(f"{name:<22} {p['breathing_rate']:>5} {p['heart_rate']:>5} "
          f"{p['breathing_regularity']:>7.2f} {p['hrv']:>5.2f} "
          f"{p['movement_amplitude']:>6.2f} {p['movement_regularity']:>6.2f} {status:>10}")

## 4. PUEK: The Room as Your Key

A **Physical Unclonable Environment Key** (PUEK) exploits the fact that every
room has a unique electromagnetic signature. The CSI covariance matrix:

$$\mathbf{R} = \frac{1}{T} \sum_{t=1}^{T} \mathbf{h}(t) \mathbf{h}(t)^H$$

where $\mathbf{h}(t) \in \mathbb{C}^{56}$ is the CSI vector at time $t$, encodes
the multipath structure of the room. The eigenvalue decomposition
$\mathbf{R} = \mathbf{U} \mathbf{\Lambda} \mathbf{U}^H$ yields eigenvalues
$\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_{56}$ that form the room's
fingerprint.

Properties that make PUEK useful for security:

1. **Stability**: The top eigenvalues are dominated by walls and large surfaces,
   which do not move. They are stable across hours and days.
2. **Uniqueness**: No two rooms have the same geometry, materials, and furniture
   arrangement. The eigenvalue spectrum is as unique as a fingerprint.
3. **Unclonability**: To spoof a PUEK, an attacker would need to physically
   reconstruct the room's geometry and materials to sub-centimeter precision.
4. **Liveness**: The eigenvalues shift when people enter/exit, providing built-in
   liveness detection.

During **enrollment**, the system captures the empty-room eigenvalue spectrum and
stores it as the reference PUEK. During **verification**, the live eigenvalue
spectrum is compared against the stored reference using cosine similarity.

In [ ]:
# ── Simulate PUEK eigenvalue spectra ──
N_EIGENVALUES = 10  # Top 10 most significant

# Enrolled room: characteristic eigenvalue decay
enrolled = np.array([45.2, 28.7, 18.3, 12.1, 8.5, 5.9, 3.8, 2.4, 1.5, 0.9])

# Same room, different time: small perturbations (furniture hasn't moved)
np.random.seed(99)
same_room = enrolled + np.random.normal(0, 0.6, N_EIGENVALUES)
same_room = np.maximum(same_room, 0.1)

# Different room: completely different eigenstructure
diff_room = np.array([52.1, 22.4, 15.8, 14.2, 9.8, 7.1, 6.3, 4.0, 3.2, 2.8])

# Cosine similarity
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_same = cosine_sim(enrolled, same_room)
sim_diff = cosine_sim(enrolled, diff_room)

# ── Visualization ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7),
                                gridspec_kw={'width_ratios': [3, 2]})
fig.suptitle("PUEK: Physical Unclonable Environment Key",
             fontsize=16, fontweight='bold', color=ZM_CYAN, y=0.98)

# Left: Eigenvalue spectra comparison
x = np.arange(N_EIGENVALUES)
width = 0.25

ax1.bar(x - width, enrolled, width, label='Enrolled (reference)',
        color=ZM_CYAN, alpha=0.85, edgecolor='none')
ax1.bar(x, same_room, width, label=f'Same room, 6h later (sim={sim_same:.3f})',
        color=ZM_EMERALD, alpha=0.85, edgecolor='none')
ax1.bar(x + width, diff_room, width, label=f'Different room (sim={sim_diff:.3f})',
        color=ZM_ROSE, alpha=0.85, edgecolor='none')

ax1.set_xlabel('Eigenvalue Index', fontsize=11)
ax1.set_ylabel('Eigenvalue Magnitude', fontsize=11)
ax1.set_title('SVD Eigenvalue Spectrum', fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels([f'$\\lambda_{{{i+1}}}$' for i in range(N_EIGENVALUES)])
ax1.legend(fontsize=10)

# Right: Similarity score with clearance thresholds
scenarios = ['Same Room\n(6h later)', 'Different\nRoom']
sims = [sim_same, sim_diff]
bar_colors = [ZM_EMERALD if s > 0.95 else (ZM_AMBER if s > 0.75 else ZM_ROSE)
              for s in sims]

bars = ax2.barh(scenarios, sims, color=bar_colors, alpha=0.85,
                edgecolor='none', height=0.4)

# Threshold lines
thresholds = [(0.75, 'L1', ZM_VIOLET), (0.85, 'L2', ZM_BLUE),
              (0.95, 'L3', ZM_AMBER), (0.98, 'L4', ZM_ROSE)]
for thresh, label, color in thresholds:
    ax2.axvline(x=thresh, color=color, linestyle='--', alpha=0.7, linewidth=1.5)
    ax2.text(thresh, 1.35, label, ha='center', va='bottom', fontsize=10,
             color=color, fontweight='bold')

for bar, sim_val in zip(bars, sims):
    ax2.text(sim_val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{sim_val:.4f}', va='center', fontsize=12, fontweight='bold',
             color='#e2e8f0')

ax2.set_xlim(0, 1.1)
ax2.set_xlabel('Cosine Similarity', fontsize=11)
ax2.set_title('PUEK Verification Score', fontsize=13)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print(f"\nPUEK Verification Results:")
print(f"  Same room (6h later):  similarity = {sim_same:.4f}", end='  -> ')
for thresh, label, _ in reversed(thresholds):
    if sim_same >= thresh:
        print(f"PASSES {label} threshold ({thresh})")
        break
print(f"  Different room:        similarity = {sim_diff:.4f}", end='  -> ')
passed = False
for thresh, label, _ in reversed(thresholds):
    if sim_diff >= thresh:
        print(f"PASSES {label} threshold ({thresh})")
        passed = True
        break
if not passed:
    print(f"FAILS all thresholds (below L1={thresholds[0][0]})")

## 5. Clearance Level Enforcement

Q-Mesh implements four clearance levels, each layering additional cryptographic
primitives:

| Level | Name | Requirements | Use Case |
|-------|------|-------------|----------|
| L1 | Basic | PUEK room match | Office access, meeting rooms |
| L2 | Standard | L1 + biometric profile match | Workstations, file access |
| L3 | High | L2 + vital sign validation + duress check | Server rooms, financial systems |
| L4 | Critical | L3 + EM Canary sweep + topology lock | SCIF, nuclear, command centers |

Each level is a strict superset of the previous one. The system never
downgrades; if a higher-level check fails, the session is terminated even if
lower-level checks passed. This follows the principle of **fail-closed
security**: ambiguity is resolved in favor of denial.

In [ ]:
# ── Clearance Level Dashboard ──
scenarios = [
    {
        'title': 'L1: Employee enters office',
        'level': 'L1',
        'checks': [
            ('PUEK Room Match', True, '0.97'),
            ('Biometric Match', None, 'N/A'),
            ('Vital Signs', None, 'N/A'),
            ('EM Canary', None, 'N/A'),
            ('Topology Lock', None, 'N/A'),
        ],
        'result': 'ACCESS GRANTED',
        'result_color': ZM_EMERALD,
    },
    {
        'title': 'L2: + Biometric confirmed',
        'level': 'L2',
        'checks': [
            ('PUEK Room Match', True, '0.97'),
            ('Biometric Match', True, '0.94'),
            ('Vital Signs', None, 'N/A'),
            ('EM Canary', None, 'N/A'),
            ('Topology Lock', None, 'N/A'),
        ],
        'result': 'ACCESS GRANTED',
        'result_color': ZM_EMERALD,
    },
    {
        'title': 'L3: Stress detected!',
        'level': 'L3',
        'checks': [
            ('PUEK Room Match', True, '0.96'),
            ('Biometric Match', True, '0.91'),
            ('Vital Signs', False, 'HR=108, DURESS'),
            ('EM Canary', None, 'Skipped'),
            ('Topology Lock', None, 'Skipped'),
        ],
        'result': 'SESSION TERMINATED',
        'result_color': ZM_ROSE,
    },
    {
        'title': 'L4: Hidden device found!',
        'level': 'L4',
        'checks': [
            ('PUEK Room Match', True, '0.98'),
            ('Biometric Match', True, '0.96'),
            ('Vital Signs', True, 'HR=65, OK'),
            ('EM Canary', False, 'RF anomaly!'),
            ('Topology Lock', None, 'Skipped'),
        ],
        'result': 'KEYS DESTROYED',
        'result_color': ZM_ROSE,
    },
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Clearance Level Enforcement Dashboard",
             fontsize=16, fontweight='bold', color=ZM_CYAN, y=0.98)

for idx, (ax, scenario) in enumerate(zip(axes.flat, scenarios)):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.set_title(scenario['title'], fontsize=12, pad=10)

    # Check rows
    y_start = 8.5
    for i, (check_name, passed, detail) in enumerate(scenario['checks']):
        y = y_start - i * 1.3

        if passed is True:
            icon = '\u2713'
            icon_color = ZM_EMERALD
            bar_color = ZM_EMERALD
        elif passed is False:
            icon = '\u2717'
            icon_color = ZM_ROSE
            bar_color = ZM_ROSE
        else:
            icon = '\u2014'
            icon_color = '#475569'
            bar_color = '#1e293b'

        # Background bar
        rect = FancyBboxPatch((0.3, y - 0.4), 9.4, 0.9,
                              boxstyle='round,pad=0.1',
                              facecolor=bar_color, alpha=0.2,
                              edgecolor='none')
        ax.add_patch(rect)

        ax.text(1.0, y, icon, fontsize=16, fontweight='bold',
                color=icon_color, ha='center', va='center')
        ax.text(2.0, y, check_name, fontsize=10, color='#e2e8f0',
                va='center')
        ax.text(8.5, y, detail, fontsize=9, color='#94a3b8',
                va='center', ha='right')

    # Result banner
    result_rect = FancyBboxPatch((0.3, 0.3), 9.4, 1.4,
                                 boxstyle='round,pad=0.15',
                                 facecolor=scenario['result_color'],
                                 alpha=0.2,
                                 edgecolor=scenario['result_color'],
                                 linewidth=2)
    ax.add_patch(result_rect)
    ax.text(5.0, 1.0, scenario['result'], fontsize=14, fontweight='bold',
            color=scenario['result_color'], ha='center', va='center')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print("\nClearance Level Summary:")
print(f"{'Level':<6} {'Scenario':<35} {'Result':<20}")
print("-" * 62)
for s in scenarios:
    print(f"{s['level']:<6} {s['title']:<35} {s['result']:<20}")

## 6. EM Canary: Physical Intrusion Detection

The EM Canary system continuously monitors the room's eigenstructure baseline
against live CSI readings. The deviation metric is computed as:

$$\delta(t) = 1 - \text{sim}(\boldsymbol{\lambda}_{\text{baseline}}, \boldsymbol{\lambda}(t))$$

where $\text{sim}$ is cosine similarity and $\boldsymbol{\lambda}(t)$ is the current
eigenvalue vector.

Three threat zones:

- **Green** ($\delta < 5\%$): Normal operation. Small fluctuations from HVAC,
  lighting changes, minor vibrations.
- **Amber** ($5\% \leq \delta < 20\%$): Known perturbation. A person entered,
  a door opened, furniture moved. The system attempts to re-identify the source
  and re-baseline.
- **Red** ($\delta \geq 20\%$): Unknown perturbation. Could be a hidden camera,
  an RF transmitter (bug), an unauthorized person in an adjacent room, or RF
  jamming. Keys are rotated immediately and the security team is alerted.

The EM Canary catches threats that software-only systems cannot see. A hidden
camera emits RF from its wireless transmitter. A person in an adjacent
thin-walled room perturbs the EM field through the wall. An RF jammer creates
massive deviation. None of these are visible to a firewall or IDS.

In [ ]:
# ── Simulate 60-second EM Canary timeline ──
T_CANARY = 60
FS_CANARY = 10
N_CANARY = T_CANARY * FS_CANARY
t_canary = np.linspace(0, T_CANARY, N_CANARY)

np.random.seed(77)

# Build deviation timeline
deviation = np.ones(N_CANARY) * 2.5 + np.random.normal(0, 0.8, N_CANARY)

# Event 1: Person enters room (t=20-25s)
mask1 = (t_canary >= 20) & (t_canary < 25)
deviation[mask1] = 15 + np.random.normal(0, 2, mask1.sum())

# Event 2: Person identified, re-baselined (t=25-30s)
mask2 = (t_canary >= 25) & (t_canary < 30)
deviation[mask2] = np.linspace(12, 3, mask2.sum()) + np.random.normal(0, 0.5, mask2.sum())

# Normal (t=30-35s)
mask3 = (t_canary >= 30) & (t_canary < 35)
deviation[mask3] = 3.0 + np.random.normal(0, 0.8, mask3.sum())

# Event 3: Unknown RF device detected (t=35-40s)
mask4 = (t_canary >= 35) & (t_canary < 40)
deviation[mask4] = 40 + np.random.normal(0, 3, mask4.sum())

# Event 4: Keys rotated, elevated alert (t=40-45s)
mask5 = (t_canary >= 40) & (t_canary < 45)
deviation[mask5] = np.linspace(35, 25, mask5.sum()) + np.random.normal(0, 2, mask5.sum())

# Event 5: Device removed, return to normal (t=45-50s)
mask6 = (t_canary >= 45) & (t_canary < 50)
deviation[mask6] = np.linspace(20, 4, mask6.sum()) + np.random.normal(0, 1, mask6.sum())

# Normal with new keys (t=50-60s)
mask7 = (t_canary >= 50)
deviation[mask7] = 3.5 + np.random.normal(0, 0.8, mask7.sum())

deviation = np.clip(deviation, 0, 55)

# Threat level (step function)
threat_level = np.zeros(N_CANARY)
threat_level[(t_canary >= 20) & (t_canary < 25)] = 1
threat_level[(t_canary >= 25) & (t_canary < 30)] = 0.5
threat_level[(t_canary >= 35) & (t_canary < 45)] = 2
threat_level[(t_canary >= 45) & (t_canary < 50)] = 1

# ── Visualization ──
fig, ax1 = plt.subplots(figsize=(16, 7))
fig.suptitle("EM Canary: Real-Time Physical Intrusion Detection",
             fontsize=16, fontweight='bold', color=ZM_CYAN, y=0.98)

# Background zones
ax1.axhspan(0, 5, color=ZM_EMERALD, alpha=0.06, label='Green zone (<5%)')
ax1.axhspan(5, 20, color=ZM_AMBER, alpha=0.06, label='Amber zone (5-20%)')
ax1.axhspan(20, 55, color=ZM_ROSE, alpha=0.06, label='Red zone (>20%)')

# Deviation line
ax1.plot(t_canary, deviation, color=ZM_CYAN, linewidth=1.8, alpha=0.9,
         label='EM Deviation (%)', zorder=5)

# Threshold lines
ax1.axhline(y=5, color=ZM_AMBER, linestyle='--', alpha=0.5, linewidth=1)
ax1.axhline(y=20, color=ZM_ROSE, linestyle='--', alpha=0.5, linewidth=1)

# Event annotations
events = [
    (22.5, 18, 'Person enters\nroom', ZM_AMBER),
    (27.5, 8, 'Biometric ID\nconfirmed', ZM_EMERALD),
    (37.5, 48, 'Unknown RF\ndevice detected!', ZM_ROSE),
    (42.5, 32, 'Keys rotated\nalert raised', ZM_AMBER),
    (47.5, 15, 'Device removed', ZM_EMERALD),
    (55, 5, 'Normal ops\n(new keys)', ZM_EMERALD),
]
for ex, ey, etxt, ecolor in events:
    ax1.annotate(etxt, xy=(ex, ey), fontsize=8, fontweight='bold',
                 color=ecolor, ha='center',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='#111827',
                           edgecolor=ecolor, alpha=0.9))

ax1.set_xlabel('Time (seconds)', fontsize=12)
ax1.set_ylabel('EM Field Deviation (%)', fontsize=12)
ax1.set_xlim(0, 60)
ax1.set_ylim(0, 55)
ax1.legend(fontsize=9, loc='upper left')

# Secondary y-axis: Threat level
ax2 = ax1.twinx()
ax2.fill_between(t_canary, threat_level, step='mid', alpha=0.15, color=ZM_ROSE)
ax2.plot(t_canary, threat_level, color=ZM_ROSE, linewidth=1.5, alpha=0.5,
         drawstyle='steps-mid', label='Threat Level')
ax2.set_ylabel('Threat Level', fontsize=12, color=ZM_ROSE)
ax2.set_yticks([0, 1, 2])
ax2.set_yticklabels(['Normal', 'Elevated', 'Critical'], fontsize=9, color=ZM_ROSE)
ax2.set_ylim(-0.2, 3)
ax2.tick_params(axis='y', colors=ZM_ROSE)
ax2.legend(fontsize=9, loc='upper right')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print("\nEM Canary Event Log:")
print(f"{'Time':>6} {'Event':<35} {'Deviation':>10} {'Action':<20}")
print("-" * 75)
log_entries = [
    ('0-20s',  'Normal operation',                '2-5%',    'None'),
    ('20-25s', 'Person enters room',              '~15%',    'Bio-ID triggered'),
    ('25-30s', 'Person identified, re-baseline',  '12->3%',  'Re-baselined'),
    ('30-35s', 'Normal',                          '~3%',     'None'),
    ('35-40s', 'Unknown RF device detected',      '~40%',    'CRITICAL ALERT'),
    ('40-45s', 'Response: keys rotated',          '25-35%',  'Key rotation'),
    ('45-50s', 'Device removed, normalizing',     '20->4%',  'De-escalation'),
    ('50-60s', 'Normal with new keys',            '~3.5%',   'Monitoring'),
]
for time_s, event, dev, action in log_entries:
    print(f"{time_s:>6} {event:<35} {dev:>10} {action:<20}")

## 7. The Complete Zero-Interaction Auth Pipeline

From the user's perspective, they simply walk into the room. There is no login
screen, no badge tap, no fingerprint reader. Behind the scenes, six
cryptographic primitives work in concert:

1. **CSI Entropy**: Raw phase data from WiFi subcarriers seeds the session key
   material. This entropy is information-theoretically unpredictable because it
   depends on the exact multipath environment at the exact moment of capture.

2. **PUEK Verification**: The room's eigenvalue spectrum is compared against the
   enrolled reference. This confirms the session is occurring in an authorized
   physical location.

3. **Vital Authentication**: Breathing rate, heart rate, and micro-movement
   patterns are extracted from CSI and matched against the enrolled user's
   biometric profile. Continuous, not one-shot.

4. **EM Canary**: The room's EM field is continuously monitored for anomalies
   that indicate physical intrusion, surveillance devices, or RF jamming.

5. **Topology Authentication**: The network graph of all RuView nodes is locked.
   Any node that disappears, appears, or changes position triggers
   re-authentication.

6. **Spatiotemporal Signatures**: Every authentication event is signed with a
   timestamp and location hash, providing legal non-repudiation for audit
   trails (DORA Art. 7 compliance).

No passwords. No badges. No biometric scanners. The physics does the work.

In [ ]:
# ── Full Pipeline Visualization ──
fig, ax = plt.subplots(figsize=(18, 9))
fig.suptitle("Zero-Interaction Authentication Pipeline",
             fontsize=16, fontweight='bold', color=ZM_CYAN, y=0.97)

ax.set_xlim(-1, 18)
ax.set_ylim(-1, 10)
ax.set_xticks([])
ax.set_yticks([])
ax.grid(False)

# Pipeline stages
stages = [
    {'name': 'Person\nEnters Room', 'x': 0.5, 'time': '0.0s',
     'metric': '', 'color': '#475569'},
    {'name': 'CSI\nCaptured', 'x': 3.0, 'time': '0.1s',
     'metric': '56 subcarriers', 'color': ZM_BLUE},
    {'name': 'Entropy\nExtracted', 'x': 5.5, 'time': '0.5s',
     'metric': '7.98 bits/byte', 'color': ZM_VIOLET},
    {'name': 'PUEK\nVerified', 'x': 8.0, 'time': '1.2s',
     'metric': 'Sim: 0.974', 'color': ZM_CYAN},
    {'name': 'Biometrics\nMatched', 'x': 10.5, 'time': '0.8s',
     'metric': 'HR: 68 BPM', 'color': ZM_AMBER},
    {'name': 'Session\nActive', 'x': 13.0, 'time': '0.3s',
     'metric': 'L3 Granted', 'color': ZM_EMERALD},
    {'name': 'Continuous\nMonitoring', 'x': 15.5, 'time': 'ongoing',
     'metric': 'EM Canary ON', 'color': ZM_EMERALD},
]

box_w = 2.0
box_h = 2.8
y_center = 6.5

for i, stage in enumerate(stages):
    x = stage['x']
    y = y_center

    # Box
    rect = FancyBboxPatch((x - box_w/2, y - box_h/2), box_w, box_h,
                          boxstyle='round,pad=0.15',
                          facecolor='#111827',
                          edgecolor=stage['color'],
                          linewidth=2, alpha=0.95)
    ax.add_patch(rect)

    # Stage name
    ax.text(x, y + 0.5, stage['name'], ha='center', va='center',
            fontsize=10, fontweight='bold', color=stage['color'])

    # Time
    ax.text(x, y - 0.4, stage['time'], ha='center', va='center',
            fontsize=9, color='#94a3b8')

    # Metric
    if stage['metric']:
        ax.text(x, y - 0.9, stage['metric'], ha='center', va='center',
                fontsize=8, color='#e2e8f0', style='italic')

    # Checkmark above box
    if i > 0:
        ax.text(x, y + box_h/2 + 0.3, '\u2713', ha='center', va='center',
                fontsize=16, fontweight='bold', color=ZM_EMERALD)

    # Arrow to next stage
    if i < len(stages) - 1:
        next_x = stages[i + 1]['x']
        arrow = FancyArrowPatch(
            (x + box_w/2 + 0.05, y),
            (next_x - box_w/2 - 0.05, y),
            arrowstyle='->', mutation_scale=15,
            color='#475569', linewidth=2)
        ax.add_patch(arrow)

# Summary box at bottom
summary_rect = FancyBboxPatch((1.0, 0.5), 15.0, 2.8,
                              boxstyle='round,pad=0.2',
                              facecolor='#111827',
                              edgecolor=ZM_EMERALD,
                              linewidth=2, alpha=0.9)
ax.add_patch(summary_rect)

summary_lines = [
    ('CLEARANCE GRANTED: L3 -- High', ZM_EMERALD, 13, True),
    ('Authentication time: 2.9 seconds  |  Active primitives: '
     'PUEK + Vital Auth + EM Canary', '#e2e8f0', 10, False),
    ('Session key: [REDACTED] (rolling HMAC-SHA3-256, 30s refresh)  |  '
     'Entropy source: CSI + QRNG pool', '#94a3b8', 9, False),
]

for i, (text, color, size, bold) in enumerate(summary_lines):
    ax.text(8.5, 2.8 - i * 0.8, text, ha='center', va='center',
            fontsize=size, color=color,
            fontweight='bold' if bold else 'normal')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print("\nPipeline Timing Breakdown:")
print(f"{'Stage':<22} {'Duration':>10} {'Cumulative':>12}")
print("-" * 48)
cumulative = 0.0
timings = [0.0, 0.1, 0.5, 1.2, 0.8, 0.3]
stage_names = ['Person enters', 'CSI captured', 'Entropy extracted',
               'PUEK verified', 'Biometrics matched', 'Session active']
for name, dur in zip(stage_names, timings):
    cumulative += dur
    print(f"{name:<22} {dur:>9.1f}s {cumulative:>11.1f}s")
print(f"\nTotal authentication time: {cumulative:.1f}s")
print(f"Continuous monitoring: EM Canary + Vital Auth (ongoing, no user action)")

## Summary

---

### Key Takeaways

**1. WiFi CSI is an unclonable security substrate.** The electromagnetic
eigenstructure of a room is determined by geometry, materials, and contents at
sub-wavelength precision. You cannot fake it without physically reconstructing
the entire environment.

**2. Biometrics from WiFi signals are contactless, continuous, and
coercion-aware.** Breathing rate and heart rate extracted from CSI phase shifts
provide identity verification that works through walls, requires no hardware on
the person, and detects stress/duress automatically.

**3. Clearance levels L1-L4 layer primitives for progressive security.** Each
level adds cryptographic checks. L1 verifies the room. L2 adds biometric
identity. L3 validates vital signs and detects coercion. L4 sweeps for
physical surveillance devices and locks the network topology.

**4. The system requires zero user interaction.** Authentication happens by
physics. Walk into the room, the room authenticates you. No passwords, no
badges, no biometric scanners. The ESP32-S3 nodes do all the sensing at 20 Hz.

**5. EM Canary detects physical intrusions that software-only systems cannot
see.** Hidden cameras, RF bugs, unauthorized presence in adjacent rooms, and RF
jamming all create measurable perturbations in the CSI eigenstructure. Firewalls
and IDS are blind to these threats; Q-Mesh is not.

---

> All data in this notebook is synthetically generated. Production deployment
> requires [RuView ESP32-S3 hardware](https://github.com/MoHoushmand/RuView)
> for real CSI data acquisition.

**Next notebook:** [09 - Deployment & Operations](09_deployment_ops.ipynb)